# Get monthly data from Ed's Rainfall Rescue project

A key part of processing the daily rainfall data is to match it to the monthly rainfall data already digitised by Rainfall Rescue. The monthly data already has station locations and other metadata, so if we can match a daily record to a monthly one, we can use the metadata from the monthly data to enrich the daily data.

Start by cloning the Rainfall Rescue GitHub repository:

In [2]:
# Setup

import os
import subprocess

# Target location for the Rainfall Rescue repository
repo_dir = f"{os.getenv('PDIR')}/Rainfall-Rescue"

# Repository URL for Rainfall Rescue
repo_url = "git@github.com:ed-hawkins/rainfall-rescue.git"

In [2]:
# Clone the Rainfall Rescue repository if it doesn't already exist

if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", repo_url, repo_dir])

Cloning into '/data/scratch/philip.brohan/ADRQ/Rainfall-Rescue'...
Updating files: 100% (123722/123722), done.


## Build SQLite database from combined station CSV files

This performs a full rebuild of the SQLite database from combined files in `DATA` (excluding `TYRain_*.csv` source sheets).

In [3]:
from pathlib import Path

from src.rainfall_rescue_sqlite.ingest import default_db_path, ingest_combined_csvs

rainfall_rescue_root = Path(repo_dir)
db_path = default_db_path()

result = ingest_combined_csvs(
    rainfall_rescue_root=rainfall_rescue_root,
    db_path=db_path,
)

result

IngestionResult(db_path=PosixPath('/data/scratch/philip.brohan/ADRQ/Rainfall-Rescue/rainfall_rescue.sqlite'), source_root=PosixPath('/data/scratch/philip.brohan/ADRQ/Rainfall-Rescue/DATA'), files_discovered=8690, files_ingested=8690, station_rows=8690, monthly_rows=3374861, annual_rows=277250, errors=0)

In [4]:
import sqlite3

with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    station_count = cur.execute("SELECT COUNT(*) FROM stations").fetchone()[0]
    monthly_count = cur.execute("SELECT COUNT(*) FROM monthly_rainfall").fetchone()[0]
    annual_count = cur.execute("SELECT COUNT(*) FROM annual_totals").fetchone()[0]

    print(f"Stations: {station_count}")
    print(f"Monthly rows: {monthly_count}")
    print(f"Annual rows: {annual_count}")

    sample = cur.execute(
        """
        SELECT s.location_name, m.year, m.month, m.rainfall_in
        FROM monthly_rainfall m
        JOIN stations s ON s.station_file_id = m.station_file_id
        WHERE s.location_name LIKE 'ABERPORTH%'
        ORDER BY m.year, m.month
        LIMIT 24
        """
    ).fetchall()

sample[:5]

Stations: 8690
Monthly rows: 3374861
Annual rows: 277250


[('ABERPORTH', 1941, 1, 2.94),
 ('ABERPORTH', 1941, 2, 2.72),
 ('ABERPORTH', 1941, 3, 3.0),
 ('ABERPORTH', 1941, 4, 1.33),
 ('ABERPORTH', 1941, 5, 2.02)]

In [5]:
# Retrieve station metadata for a few stations

with sqlite3.connect(db_path) as conn:
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    stations = cur.execute(
        """
        SELECT
            station_file_id,
            location_name,
            station_number,
            grid_reference,
            longitude,
            latitude,
            elevation_ft,
            station_file_name,
            source_path
        FROM stations
        ORDER BY location_name
        LIMIT 5
        """
    ).fetchall()

for row in stations:
    print(
        f"{row['location_name']!s:<40}"
        f"  stn={row['station_number'] or 'n/a':<12}"
        f"  lat={row['latitude'] or 'n/a'!s:<10}"
        f"  lon={row['longitude'] or 'n/a'!s:<12}"
        f"  elev={row['elevation_ft'] or 'n/a'!s:<6} ft"
        f"  grid={row['grid_reference'] or 'n/a'}"
    )

1 MI FROM EDINBURGH                       stn=RR502         lat=55.94251    lon=-3.187925     elev=250    ft  grid=NT259728
63 BLOOMSBURY ST BIRMINGHAM               stn=RR1550        lat=52.4922     lon=-1.8731       elev=340    ft  grid=SP087882
ABBEY ST BATHANS, BERWICKSHIRE            stn=RR2226        lat=55.852382   lon=-2.3897074    elev=450    ft  grid=NT757622
ABBEY-CWMHIR-THE-HALL                     stn=4241/5        lat=52.330722   lon=-3.3881959    elev=877    ft  grid=SO055712
ABBEY-LEIX-BLANDSFORT                     stn=n/a           lat=52.930139   lon=-7.284667     elev=532    ft  grid=IS4814786802
